# Solution Walkthrough: PPE Compliance Detection Pipeline

**Instructor-led** | End of Day 2 | ~35 minutes

This notebook shows one complete solution to the workshop challenge.
Your approach may differ -- that's the point. There are many valid paths.

**Pipeline overview:**

```
Auto-label (SAM3)  -->  Curate (filter)  -->  Train (YOLO26n)  -->  Evaluate  -->  Compliance
    teacher               data curation         student model       mAP + business    post-processing
```

**What we reveal here:**
1. The complete pipeline, end to end, with every script call and every result
2. The prompt engineering discovery (2.2x more helmets with minimal prompt)
3. The compliance postprocessor -- translating detections into safety assessments
4. FiftyOne data-centric analysis -- finding label errors, redundant images, and failure modes
5. The five key insights from our experiments

> **Instructor note:** This notebook is pre-run with all outputs cached.
> Walk through each section, pausing at the marked discussion points.
> The goal is not to re-run -- it is to reveal the reasoning behind each step.

In [ ]:
import sys, subprocess, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────────────────────
WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
IMAGES_DIR = DATA_DIR / "synthetic_ppe"
LABELED_DIR = DATA_DIR / "ppe_dataset_exp_a"          # raw auto-labels
FILTERED_DIR = DATA_DIR / "ppe_dataset_exp_a_filtered" # after tiny-label filtering
RESULTS_DIR = DATA_DIR / "ppe_results"

print(f"Workshop:  {WORKSHOP_DIR}")
print(f"Images:    {IMAGES_DIR}  ({'exists' if IMAGES_DIR.exists() else 'MISSING'})")
print(f"Labels:    {LABELED_DIR}  ({'exists' if LABELED_DIR.exists() else 'MISSING'})")
print(f"Filtered:  {FILTERED_DIR}  ({'exists' if FILTERED_DIR.exists() else 'MISSING'})")
print(f"Results:   {RESULTS_DIR}  ({'exists' if RESULTS_DIR.exists() else 'MISSING'})")

---
## Part 1: Auto-Labeling — Prompt Sensitivity

The most impactful discovery in this workshop: **prompt choice is a first-order lever**.

Different wording finds different objects. In foundation model experiments,
to `"helmet. person."` found **2.2x more helmets** — same model, same images, different words.

SAM3 uses a different API: it takes **one concept per call** (not period-separated strings).
The question is: does the same kind of prompt sensitivity apply?
Let's test different single-concept prompts for helmet detection and find out.

> **Key lesson from Day 1:** Negation doesn't work. You cannot ask a model to detect
> "person without helmet" — the model attends to the nouns (person, helmet) and ignores
> the negation. This is why we use 2-class detection + post-processing code.

### Prompt Engineering: Single-Concept Comparison

SAM3 HF takes **one concept per call** — the labeling script runs separate calls
for `"hard hat"` and `"person"`. Let's test whether the *word choice* for the helmet
concept matters, and verify that negative prompts fail.

We test four helmet prompts on the same images:
1. `"helmet"` — broad, simple noun
2. `"hard hat"` — compound noun (current default in our labeling script)
3. `"safety helmet"` — verbose synonym
4. `"person not wearing a hard hat"` — **negative prompt** (will it detect absence?)

In [ ]:
# ── SAM3 HF Prompt Comparison: Single-Concept Helmet Detection ────────────────
# Each prompt is a SINGLE concept sent to SAM3 HF in a separate call.
# This is how the labeling script works: one concept → one set of boxes.

from transformers import Sam3Model, Sam3Processor
import torch

# Auto-detect device
if torch.cuda.is_available():
    _device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    _device = "mps"
else:
    _device = "cpu"

print(f"Loading SAM3 on {_device}...")
sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")
sam3_model = Sam3Model.from_pretrained("facebook/sam3").to(_device)
print("SAM3 loaded.")

# ── Pick 3 diverse sample images ──────────────────────────────────────────────
sample_images = []
for cat in ["mixed_compliance", "close_up", "highway"]:
    cat_dir = IMAGES_DIR / cat
    if cat_dir.exists():
        imgs = sorted(cat_dir.glob("*.*"))
        if imgs:
            sample_images.append(imgs[0])
if len(sample_images) < 3:
    all_imgs = sorted(IMAGES_DIR.rglob("*.jpg")) + sorted(IMAGES_DIR.rglob("*.png"))
    sample_images = all_imgs[:3]

print(f"Sample images: {[p.name for p in sample_images]}")

# ── Single-concept prompts to compare ────────────────────────────────────────
# Each is sent as ONE text string in a separate SAM3 HF call.
HELMET_PROMPTS = [
    ("helmet",                         "#2ecc71", "broad"),
    ("hard hat",                       "#3498db", "default"),
    ("safety helmet",                  "#f39c12", "verbose"),
    ("person not wearing a hard hat",  "#e74c3c", "NEGATIVE"),
]


def run_sam3_single(img, text, threshold=0.5, mask_threshold=0.5):
    """Run SAM3 HF with a SINGLE concept prompt. Returns list of [x1,y1,x2,y2] boxes."""
    inputs = sam3_processor(images=img, text=text, return_tensors="pt").to(_device)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    target_sizes = inputs.get("original_sizes")
    if target_sizes is not None:
        target_sizes = target_sizes.tolist()
    else:
        target_sizes = [(img.size[1], img.size[0])]
    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=target_sizes,
    )[0]
    boxes = results.get("boxes", [])
    if hasattr(boxes, "tolist"):
        boxes = boxes.tolist()
    return boxes or []


# ── Run all prompts on all images → visual grid ──────────────────────────────
n_imgs = len(sample_images)
n_prompts = len(HELMET_PROMPTS)

fig, axes = plt.subplots(n_imgs, n_prompts, figsize=(5 * n_prompts, 5 * n_imgs))
if n_imgs == 1:
    axes = [axes]

total_per_prompt = {p[0]: 0 for p in HELMET_PROMPTS}

for row, img_path in enumerate(sample_images):
    img = Image.open(img_path).convert("RGB")
    for col, (prompt_text, color, label) in enumerate(HELMET_PROMPTS):
        ax = axes[row][col]
        ax.imshow(np.array(img))

        boxes = run_sam3_single(img, prompt_text)
        total_per_prompt[prompt_text] += len(boxes)

        for box in boxes:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)

        if row == 0:
            ax.set_title(f'"{prompt_text}"\n({label}) — {len(boxes)} boxes',
                         fontsize=9, fontweight="bold", color=color)
        else:
            ax.set_title(f"{len(boxes)} boxes", fontsize=10, color=color)
        ax.axis("off")

    axes[row][0].set_ylabel(img_path.parent.name, fontsize=11, fontweight="bold",
                            rotation=0, labelpad=80, va="center")

fig.suptitle("SAM3 HF: Helmet Detection with Different Single-Concept Prompts",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Also compare PERSON concept prompts ───────────────────────────────────
PERSON_PROMPTS = [
    ("person",              "#2ecc71"),
    ("worker",              "#3498db"),
    ("construction worker", "#f39c12"),
    ("human",               "#9b59b6"),
]

print("\n--- Person concept comparison ---")
fig2, axes2 = plt.subplots(1, n_prompts_p := len(PERSON_PROMPTS),
                           figsize=(5 * n_prompts_p, 5))
# Use first sample image
img = Image.open(sample_images[0]).convert("RGB")
person_totals = {}

for col, (prompt_text, color) in enumerate(PERSON_PROMPTS):
    ax = axes2[col]
    ax.imshow(np.array(img))
    boxes = run_sam3_single(img, prompt_text)
    person_totals[prompt_text] = len(boxes)
    for box in boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
    ax.set_title(f'"{prompt_text}" — {len(boxes)} boxes',
                 fontsize=10, fontweight="bold", color=color)
    ax.axis("off")

fig2.suptitle(f"Person Concept Comparison: {Path(sample_images[0]).name}",
              fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Aggregate summary ────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"HELMET PROMPT COMPARISON — {n_imgs} images, SAM3 HF")
print("=" * 65)
for prompt_text, color, label in HELMET_PROMPTS:
    count = total_per_prompt[prompt_text]
    tag = ""
    if label == "NEGATIVE":
        tag = " ← finds persons, NOT absence"
    print(f'  "{prompt_text:<36s}"  {count:>3d} detections{tag}')
print("=" * 65)

print(f"\nPERSON PROMPT COMPARISON — 1 image")
print("-" * 50)
for prompt_text, color in PERSON_PROMPTS:
    count = person_totals[prompt_text]
    print(f'  "{prompt_text:<24s}"  {count:>3d} detections')
print("-" * 50)

print("\n★ NEGATIVE PROMPT: 'person not wearing a hard hat' does NOT detect absence.")
print("  It detects PERSONS (the visible features). The model cannot see 'without'.")
print("  → Golden rule: detect THINGS with models, check RELATIONSHIPS with code.")

print("\n★ PROMPT SENSITIVITY varies by model and by concept. The effect can be")
print("  'hard hat' by 2.2x. SAM3 HF shows less variation here — but the lesson")
print("  stands: always test your prompts. Never assume.")

# Clean up model to free GPU memory for training
del sam3_model, sam3_processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("\nSAM3 model unloaded (memory freed for training).")


In [ ]:
# ── Step 1: Auto-label with SAM3 HF ──────────────────────────────────────────
# The auto-labeler sends SEPARATE single-concept prompts to SAM3 HF:
#   - "hard hat"  → class 0 (hardhat detections)
#   - "person"    → class 1 (person detections)
#
# You can experiment with different prompts by editing MODE_CONFIG in the script,
# or by modifying the --mode argument. Available modes:
#   exp_a: hardhat + person (2-class, recommended for compliance)
#   3class: hardhat + no_hardhat + person (3-class, negation problems)
#   exp_b: safe + unsafe (direct compliance labels)
#
# ~5-8 minutes on A40 GPU, ~15-20 minutes on MPS

autolabel_script = DATA_DIR / "auto_label_sam3_hf.py"

# ── Configurable prompts ─────────────────────────────────────────────────────
LABELING_MODE = "exp_a"  # Change to experiment with different modes
RELABEL = False          # Flip to True to force re-labeling (overwrites existing)

import shutil

if RELABEL and LABELED_DIR.exists():
    print(f"RELABEL=True → removing existing labels: {LABELED_DIR}")
    shutil.rmtree(LABELED_DIR)

if not LABELED_DIR.exists():
    print(f"Running auto-labeling with SAM3 HF (mode={LABELING_MODE})...")
    print(f"  Prompts: hardhat='hard hat', person='person'")
    print(f"  Output: {LABELED_DIR}")
    subprocess.run([
        sys.executable, str(autolabel_script),
        "--mode", LABELING_MODE,
        "--source-dir", str(IMAGES_DIR),
        "--output-dir", str(LABELED_DIR),
    ], check=True)
    print("Auto-labeling complete.")
else:
    print(f"Auto-labels already exist: {LABELED_DIR}")
    import yaml
    yaml_path = LABELED_DIR / "dataset.yaml"
    class_names = {}
    if yaml_path.exists():
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        class_names = cfg.get("names", {})

    for split in ["train", "val"]:
        labels_dir = LABELED_DIR / "labels" / split
        if not labels_dir.exists():
            continue
        counts = {}
        for lf in sorted(labels_dir.glob("*.txt")):
            for line in lf.read_text().strip().splitlines():
                if not line.strip():
                    continue
                cls_id = int(line.split()[0])
                name = class_names.get(cls_id, f"class_{cls_id}")
                counts[name] = counts.get(name, 0) + 1
        print(f"  {split}: {counts}")

### Visualize raw auto-labels

Before any curation, let's see what the teacher model produced.
Look for tiny noise labels, missed objects, and incorrect class assignments.

In [ ]:
# ── Utility: show YOLO labels on an image ────────────────────────────────────

CLASS_COLORS = {
    0: (0.0, 0.78, 0.0),     # hardhat -- green
    1: (0.86, 0.12, 0.12),   # person -- red
}
CLASS_NAMES = {0: "hardhat", 1: "person"}


def show_labeled_image(image_path, label_path, figsize=(12, 8), title=None):
    """Display an image with YOLO-format bounding boxes overlaid."""
    img = np.array(Image.open(image_path))
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)

    if Path(label_path).exists():
        for line in Path(label_path).read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)

            color = CLASS_COLORS.get(cls_id, (0.8, 0.8, 0.8))
            name = CLASS_NAMES.get(cls_id, f"class_{cls_id}")

            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                 linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, name, color=color, fontsize=8, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))

    ax.set_title(title or Path(image_path).name)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


# Show 4 validation images with their auto-generated labels
val_images = sorted((LABELED_DIR / "images" / "val").glob("*.*"))
print(f"Validation images: {len(val_images)}")

for img_path in val_images[:4]:
    label_path = LABELED_DIR / "labels" / "val" / f"{img_path.stem}.txt"
    show_labeled_image(img_path, label_path, title=f"RAW LABELS: {img_path.name}")

---
## Part 2: Data Curation

**Never train on labels you haven't inspected.**

The auto-labeler produces many tiny bounding boxes -- noise from distant objects,
reflections, or ambiguous regions. These labels are too small to be useful
(often < 15 pixels) and they actively hurt training.

**The fix:** Filter out any label where either the width or height is below 3%
of the image dimension (roughly 20 pixels at 640px input size).

**Impact:** This removes **35.6% of labels** and improves mAP50 by **+2.7%**.

> **Discussion point:** Why does removing a third of the labels *improve* the model?
> Because those labels are noise. The model wastes capacity trying to learn patterns
> that don't exist. Fewer, cleaner labels beat more, noisier labels.

In [ ]:
# ── Step 2: Filter tiny labels ────────────────────────────────────────────────
# Removes labels where width or height < 0.03 (normalized).
# This is ~20px at 640px input -- too small to be a real detection.

filter_script = DATA_DIR / "filter_tiny_labels.py"

if not FILTERED_DIR.exists():
    print("Filtering tiny labels...")
    subprocess.run([
        sys.executable, str(filter_script),
        "--input-dir", str(LABELED_DIR),
        "--output-dir", str(FILTERED_DIR),
    ], check=True)
    print("Filtering complete.")
else:
    print(f"Filtered dataset already exists: {FILTERED_DIR}")

# Compare label counts before/after filtering
def count_all_labels(dataset_dir):
    total = 0
    for split in ["train", "val"]:
        labels_dir = Path(dataset_dir) / "labels" / split
        if not labels_dir.exists():
            continue
        for lf in labels_dir.glob("*.txt"):
            lines = [l for l in lf.read_text().strip().splitlines() if l.strip()]
            total += len(lines)
    return total

if LABELED_DIR.exists() and FILTERED_DIR.exists():
    before = count_all_labels(LABELED_DIR)
    after = count_all_labels(FILTERED_DIR)
    removed = before - after
    pct = removed / before * 100 if before > 0 else 0
    print(f"\nBefore filtering: {before} labels")
    print(f"After filtering:  {after} labels")
    print(f"Removed:          {removed} labels ({pct:.1f}%)")

---
## Part 3: Training YOLO26n

With curated labels from the optimal prompt, we train a fast student model.

**Training configuration:**
- Model: `yolo26n.pt` (nano -- fast inference, small footprint)
- Epochs: 100 with patience 20 (early stopping)
- Batch size: 8
- Image size: 640
- Workers: 0 (avoids multiprocessing issues on HPC)

Training takes ~8-15 minutes on an A40 GPU.

In [ ]:
# ── Step 3: Train YOLO26n on curated labels ──────────────────────────────────

from ultralytics import YOLO
import torch

RETRAIN = False  # Flip to True to force re-training (removes solution_* dirs only)

# MPS has tensor mismatch bugs in Ultralytics 8.4.x loss computation.
# For 91 images, CPU is fast enough (~5 min). Use CUDA on HPC.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dataset_yaml = FILTERED_DIR / "dataset.yaml"
assert dataset_yaml.exists(), f"Dataset YAML not found: {dataset_yaml}"

# ── Experiment paths ─────────────────────────────────────────────────────────
EXP_640  = RESULTS_DIR / "solution_640"
EXP_1280 = RESULTS_DIR / "solution_1280"

if RETRAIN:
    for exp_dir in [EXP_640, EXP_1280]:
        if exp_dir.exists():
            print(f"RETRAIN=True → removing {exp_dir}")
            shutil.rmtree(exp_dir)

# Common training kwargs
TRAIN_KWARGS = dict(
    data=str(dataset_yaml),
    epochs=100, patience=20,
    device=DEVICE, workers=0,
)

# ── Experiment A: 640px (baseline) ───────────────────────────────────────────
if not (EXP_640 / "weights" / "best.pt").exists():
    print(f"Training Experiment A: imgsz=640 on {DEVICE}...")
    model = YOLO("yolo26n.pt")
    model.train(**TRAIN_KWARGS, batch=8, imgsz=640,
                project=str(RESULTS_DIR), name="solution_640")
    print("Experiment A (640px) complete.")
else:
    print(f"Experiment A (640px) already trained: {EXP_640}")

# ── Experiment B: 1280px (high-res) ─────────────────────────────────────────
if not (EXP_1280 / "weights" / "best.pt").exists():
    print(f"\nTraining Experiment B: imgsz=1280 on {DEVICE}...")
    model = YOLO("yolo26n.pt")
    model.train(**TRAIN_KWARGS, batch=4, imgsz=1280,
                project=str(RESULTS_DIR), name="solution_1280")
    print("Experiment B (1280px) complete.")
else:
    print(f"Experiment B (1280px) already trained: {EXP_1280}")

# ── Set MODEL_PATH to the 640 baseline (used downstream) ────────────────────
MODEL_PATH = EXP_640 / "weights" / "best.pt"
MODEL_PATH_1280 = EXP_1280 / "weights" / "best.pt"
print(f"\nMODEL_PATH (640):  {MODEL_PATH}  ({'exists' if MODEL_PATH.exists() else 'MISSING'})")
print(f"MODEL_PATH_1280:   {MODEL_PATH_1280}  ({'exists' if MODEL_PATH_1280.exists() else 'MISSING'})")

In [ ]:
# ── Training curves: compare 640 vs 1280 ────────────────────────────────────

import pandas as pd

experiments = {
    "640px": EXP_640 / "results.csv",
    "1280px": EXP_1280 / "results.csv",
}
colors = {"640px": "#2980b9", "1280px": "#e74c3c"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for label, csv_path in experiments.items():
    if not csv_path.exists():
        print(f"No results.csv for {label}: {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    c = colors[label]

    # Box loss (val)
    if "val/box_loss" in df.columns:
        axes[0].plot(df["epoch"], df["val/box_loss"], label=label, color=c)

    # mAP50
    if "metrics/mAP50(B)" in df.columns:
        axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=label, color=c, linewidth=2)
        best_idx = df["metrics/mAP50(B)"].idxmax()
        best_map = df["metrics/mAP50(B)"].max()
        axes[1].annotate(f"{label}: {best_map:.3f}",
                        xy=(df.loc[best_idx, "epoch"], best_map),
                        xytext=(10, -20 if label == "640px" else 10),
                        textcoords="offset points",
                        fontsize=10, color=c, fontweight="bold")

    # Precision & Recall
    if "metrics/precision(B)" in df.columns:
        axes[2].plot(df["epoch"], df["metrics/precision(B)"], label=f"{label} P", color=c, linestyle="-")
    if "metrics/recall(B)" in df.columns:
        axes[2].plot(df["epoch"], df["metrics/recall(B)"], label=f"{label} R", color=c, linestyle="--")

axes[0].set_title("Val Box Loss", fontsize=12)
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("mAP50", fontsize=12)
axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].set_title("Precision & Recall", fontsize=12)
axes[2].set_xlabel("Epoch"); axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.suptitle("Training Comparison: 640px vs 1280px", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Part 4: Detection Evaluation

Standard object detection metrics first, then we move to business metrics.

**Key results from our experiments:**

| Experiment | mAP50 | Hardhat AP | Person AP | Notes |
|------------|:-----:|:----------:|:---------:|-------|
| YOLOe zero-shot | ~0.50 | ~0.35 | ~0.65 | No training, slow |
| V1 (verbose prompt, raw labels) | 0.527 | 0.42 | 0.63 | Saturating curve ceiling |
| V2 (minimal prompt, raw labels) | 0.593 | 0.52 | 0.67 | Prompt breakthrough |
| **V2 + filtering** | **0.633** | **0.56** | **0.71** | **Best: prompt + curation** |

The jump from 0.527 to 0.633 comes entirely from better labeling -- same images, same model architecture.

In [ ]:
# ── Step 4: Evaluate both trained models ─────────────────────────────────────

dataset_yaml = FILTERED_DIR / "dataset.yaml"
assert dataset_yaml.exists(), f"Dataset YAML not found: {dataset_yaml}"

for label, weights in [("640px", MODEL_PATH), ("1280px", MODEL_PATH_1280)]:
    if not weights.exists():
        print(f"Skipping {label}: weights not found at {weights}")
        continue

    print(f"\n{'='*60}")
    print(f"  Evaluating: {label}  ({weights.parent.parent.name})")
    print(f"{'='*60}")

    model = YOLO(str(weights))
    metrics = model.val(
        data=str(dataset_yaml),
        conf=0.25,
        iou=0.5,
        verbose=True,
    )
    print(f"\nmAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")

    for i, ap in enumerate(metrics.box.ap50):
        name = CLASS_NAMES.get(i, f"class_{i}")
        print(f"  {name:<12} AP50 = {ap:.4f}")

# Reload the 640 model for downstream use
model = YOLO(str(MODEL_PATH))

### Data-Centric Analysis with FiftyOne

FiftyOne lets us audit our labels and model predictions at a deeper level than
aggregate metrics. The student model can actually help identify where the teacher
made mistakes -- the student audits the teacher.

**What we compute:**
- **Mistakenness**: How likely is each label to be wrong? (based on model disagreement with the label)
- **Uniqueness**: How visually distinct is each image? (helps find redundant samples)
- **TP/FP/FN analysis**: Where exactly does the model succeed and fail?

In [ ]:
# ── FiftyOne: Load dataset and add model predictions ─────────────────────────
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.utils.ultralytics as fou

# Load the filtered dataset into FiftyOne
dataset_name = "solution_walkthrough"

# Delete if it already exists (re-run safe)
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)

dataset = fo.Dataset.from_dir(
    dataset_dir=str(FILTERED_DIR),
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    name=dataset_name,
)

print(f"Loaded {len(dataset)} samples into FiftyOne")
print(dataset)

# Add model predictions
model = YOLO(str(MODEL_PATH))

for sample in dataset:
    results = model.predict(sample.filepath, conf=0.25, verbose=False)
    detections = fou.to_detections(results[0])
    sample["predictions"] = detections
    sample.save()

print(f"Added predictions to all {len(dataset)} samples")

In [ ]:
# ── FiftyOne: Evaluate and compute brain methods ─────────────────────────────

# Evaluate predictions against ground truth
eval_results = dataset.evaluate_detections(
    "predictions",
    gt_field="ground_truth",
    eval_key="eval",
    compute_mAP=True,
)

print("\nFiftyOne evaluation:")
eval_results.print_report()

# Compute mistakenness (label quality indicator)
fob.compute_mistakenness(
    dataset,
    "predictions",
    label_field="ground_truth",
)

# Compute uniqueness (visual diversity)
fob.compute_uniqueness(dataset)

print("\nBrain computations complete: mistakenness + uniqueness")

In [ ]:
# ── Error distribution: TP, FP, FN across the validation set ─────────────────

tp_counts = []
fp_counts = []
fn_counts = []

for sample in dataset:
    tp = fp = fn = 0
    if sample.predictions and sample.predictions.detections:
        for det in sample.predictions.detections:
            if hasattr(det, "eval") and det.eval:
                if det.eval == "tp":
                    tp += 1
                elif det.eval == "fp":
                    fp += 1
    if sample.ground_truth and sample.ground_truth.detections:
        for det in sample.ground_truth.detections:
            if hasattr(det, "eval") and det.eval:
                if det.eval == "fn":
                    fn += 1
    tp_counts.append(tp)
    fp_counts.append(fp)
    fn_counts.append(fn)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(tp_counts, bins=range(0, max(tp_counts or [1]) + 2), color="#27ae60", edgecolor="white")
axes[0].set_title(f"True Positives (total: {sum(tp_counts)})")
axes[0].set_xlabel("Count per image")

axes[1].hist(fp_counts, bins=range(0, max(fp_counts or [1]) + 2), color="#e74c3c", edgecolor="white")
axes[1].set_title(f"False Positives (total: {sum(fp_counts)})")
axes[1].set_xlabel("Count per image")

axes[2].hist(fn_counts, bins=range(0, max(fn_counts or [1]) + 2), color="#f39c12", edgecolor="white")
axes[2].set_title(f"False Negatives (total: {sum(fn_counts)})")
axes[2].set_xlabel("Count per image")

for ax in axes:
    ax.set_ylabel("Number of images")
    ax.grid(True, alpha=0.3)

fig.suptitle("Error Distribution Across Validation Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nSummary: {sum(tp_counts)} TP, {sum(fp_counts)} FP, {sum(fn_counts)} FN")
precision = sum(tp_counts) / max(sum(tp_counts) + sum(fp_counts), 1)
recall = sum(tp_counts) / max(sum(tp_counts) + sum(fn_counts), 1)
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}")

In [ ]:
# ── Embeddings visualization ─────────────────────────────────────────────────
# Compute image embeddings and visualize with UMAP.
# Color by: (1) scene category, (2) mistakenness, (3) uniqueness

import torch
from torchvision import transforms, models

# Extract embeddings using a pretrained ResNet18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()  # remove classification head
resnet.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

embeddings = []
categories = []
mistakenness_vals = []
uniqueness_vals = []

for sample in dataset:
    img = Image.open(sample.filepath).convert("RGB")
    tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        emb = resnet(tensor).squeeze().numpy()
    embeddings.append(emb)
    
    # Extract category from path (parent directory name)
    # Category from filename: e.g., 'cctv_cctv_01.jpg' → 'cctv'
    fname = Path(sample.filepath).stem
    parts = fname.split("_")
    cat = parts[0] if parts else "unknown"
    categories.append(cat)
    
    mistakenness_vals.append(getattr(sample, "mistakenness", 0.0) or 0.0)
    uniqueness_vals.append(getattr(sample, "uniqueness", 0.5) or 0.5)

embeddings = np.array(embeddings)

# UMAP dimensionality reduction
try:
    from umap import UMAP
    reducer = UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(embeddings)-1))
    coords = reducer.fit_transform(embeddings)
    method_name = "UMAP"
except ImportError:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=42)
    coords = reducer.fit_transform(embeddings)
    method_name = "PCA"

# Plot three views
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Color by scene category
unique_cats = sorted(set(categories))
cat_colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
cat_to_color = {c: cat_colors[i] for i, c in enumerate(unique_cats)}

for cat in unique_cats:
    mask = [c == cat for c in categories]
    idx = np.where(mask)[0]
    axes[0].scatter(coords[idx, 0], coords[idx, 1], c=[cat_to_color[cat]],
                   label=cat, s=40, alpha=0.7)
axes[0].set_title(f"{method_name}: Scene Category")
axes[0].legend(fontsize=7, loc="best", ncol=2)

# 2. Color by mistakenness
sc1 = axes[1].scatter(coords[:, 0], coords[:, 1], c=mistakenness_vals,
                      cmap="YlOrRd", s=40, alpha=0.7)
axes[1].set_title(f"{method_name}: Mistakenness (label quality)")
plt.colorbar(sc1, ax=axes[1])

# 3. Color by uniqueness
sc2 = axes[2].scatter(coords[:, 0], coords[:, 1], c=uniqueness_vals,
                      cmap="viridis", s=40, alpha=0.7)
axes[2].set_title(f"{method_name}: Uniqueness (visual diversity)")
plt.colorbar(sc2, ax=axes[2])

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Image Embedding Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Show the most "mistaken" images (likely label errors)
print("\nTop 5 most likely label errors (highest mistakenness):")
sorted_indices = np.argsort(mistakenness_vals)[::-1]
for idx in sorted_indices[:5]:
    all_samples = list(dataset)
    sample = all_samples[int(idx)]
    print(f"  {Path(sample.filepath).name}: mistakenness={mistakenness_vals[idx]:.4f}")

---
## Part 5: From Detections to Compliance (THE BIG REVEAL)

**The Problem**: Given hardhat and person detections, determine what percentage
of workers are non-compliant (not wearing a hardhat).

**Why not detect `no_hardhat` directly?**

We tried. The 3-class model (hardhat / no_hardhat / person) achieved only **25.5%
recall on `no_hardhat`**. The model is trying to detect an *absence* -- a person
without a helmet. But "without a helmet" is not a visible object. There are no
distinctive pixels that say "helmet is missing here." The model can only succeed
when ALL workers in a scene lack helmets (so the visual pattern is unambiguous),
and it fails completely when some workers wear helmets and others don't.

**The Solution**: Detect things (hardhat, person) with the model. Check relationships
with code.

**The Algorithm:**
1. For each detected **person**, extract the **HEAD REGION** (top 40% of the person bounding box)
2. For each detected **hardhat**, check if it overlaps any person's head region
3. If IoU >= 0.1 between hardhat and head region: **COMPLIANT**
4. If no hardhat overlaps: **NON-COMPLIANT**
5. Compute: `non_compliant / total_persons x 100` = compliance violation rate

> **Discussion point:** Why 40% for the head region? Why not 30% or 50%?
> It's a design choice. Workers vary in pose -- bending, reaching, sitting.
> 40% is generous enough to catch helmets even when the worker is slightly
> hunched, without being so large that a hardhat on someone's belt gets counted.
> You could tune this per deployment.

In [ ]:
# ── The Compliance Algorithm ──────────────────────────────────────────────────
# This is the core logic. Same as compliance_postprocessor.py, inlined here
# so you can see every step.

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def assess_compliance(results, conf=0.25, head_fraction=0.4, iou_threshold=0.1):
    """Assess PPE compliance from YOLO detections.

    Args:
        results: Ultralytics prediction results for one image
        conf: confidence threshold
        head_fraction: top fraction of person bbox considered as head region
        iou_threshold: minimum IoU to consider a hardhat as 'worn'

    Returns:
        dict with total_persons, compliant, non_compliant, compliance_rate,
        plus lists of person/hardhat boxes and per-person compliance flags.
    """
    boxes = results[0].boxes
    persons, hardhats = [], []

    for i, cls_id in enumerate(boxes.cls.cpu().numpy()):
        if boxes.conf[i].item() < conf:
            continue
        xyxy = boxes.xyxy[i].cpu().numpy()
        if int(cls_id) == 0:   # hardhat
            hardhats.append(xyxy)
        elif int(cls_id) == 1: # person
            persons.append(xyxy)

    compliant_flags = []
    for person in persons:
        # Extract head region (top fraction of person bbox)
        head_region = [
            person[0],                                          # x1
            person[1],                                          # y1 (top)
            person[2],                                          # x2
            person[1] + (person[3] - person[1]) * head_fraction # y2 = y1 + h * fraction
        ]

        # Check if any hardhat overlaps the head region
        is_compliant = any(
            compute_iou(head_region, hh) >= iou_threshold for hh in hardhats
        )
        compliant_flags.append(is_compliant)

    total = len(persons)
    compliant_count = sum(compliant_flags)
    non_compliant = total - compliant_count
    rate = (non_compliant / total * 100) if total > 0 else 0

    return {
        "total_persons": total,
        "compliant": compliant_count,
        "non_compliant": non_compliant,
        "compliance_violation_rate": rate,
        "persons": persons,
        "hardhats": hardhats,
        "compliant_flags": compliant_flags,
    }

print("Compliance algorithm defined.")
print("  compute_iou(box1, box2) -> float")
print("  assess_compliance(results, conf, head_fraction, iou_threshold) -> dict")

In [ ]:
# ── Compliance Visualization ──────────────────────────────────────────────────
# Color-coded bounding boxes: green=SAFE, red=UNSAFE, blue=hardhat, dashed=head region

def visualize_compliance(image_path, results, conf=0.25):
    """Visualize compliance on a single image with color-coded boxes."""
    img = np.array(Image.open(image_path))
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img)

    assessment = assess_compliance(results, conf=conf)
    persons = assessment["persons"]
    hardhats = assessment["hardhats"]
    flags = assessment["compliant_flags"]
    head_fraction = 0.4

    # Draw hardhats in blue
    for hh in hardhats:
        rect = plt.Rectangle((hh[0], hh[1]), hh[2] - hh[0], hh[3] - hh[1],
                             linewidth=2, edgecolor="#3498db", facecolor="none")
        ax.add_patch(rect)
        ax.text(hh[0], hh[1] - 3, "hardhat", color="white", fontsize=8,
                bbox=dict(facecolor="#3498db", alpha=0.8, pad=1))

    # Draw persons with compliance color
    for person, is_compliant in zip(persons, flags):
        color = "#27ae60" if is_compliant else "#e74c3c"
        label = "SAFE" if is_compliant else "UNSAFE"

        # Person box
        rect = plt.Rectangle(
            (person[0], person[1]),
            person[2] - person[0], person[3] - person[1],
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(person[0], person[1] - 3, label, color="white", fontsize=9,
                fontweight="bold",
                bbox=dict(facecolor=color, alpha=0.9, pad=1))

        # Head region (dashed)
        head_y2 = person[1] + (person[3] - person[1]) * head_fraction
        head_rect = plt.Rectangle(
            (person[0], person[1]),
            person[2] - person[0], head_y2 - person[1],
            linewidth=1, edgecolor=color, facecolor="none", linestyle="--"
        )
        ax.add_patch(head_rect)

    # Title with summary
    ax.set_title(
        f"{Path(image_path).name}  |  "
        f"Persons: {assessment['total_persons']}, "
        f"Compliant: {assessment['compliant']}, "
        f"Non-compliant: {assessment['non_compliant']} "
        f"({assessment['compliance_violation_rate']:.0f}% violation rate)",
        fontsize=11
    )
    ax.axis("off")

    # Legend
    legend_elements = [
        mpatches.Patch(edgecolor="#27ae60", facecolor="none", linewidth=2, label="Compliant (SAFE)"),
        mpatches.Patch(edgecolor="#e74c3c", facecolor="none", linewidth=2, label="Non-compliant (UNSAFE)"),
        mpatches.Patch(edgecolor="#3498db", facecolor="none", linewidth=2, label="Hardhat detection"),
        mpatches.Patch(edgecolor="gray", facecolor="none", linewidth=1,
                       linestyle="--", label="Head region (top 40%)"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()

print("Visualization function ready.")

In [ ]:
# ── Run compliance on sample images from different categories ──────────────────

model = YOLO(str(MODEL_PATH))

# Pick images that showcase different scenarios
sample_categories = ["easy", "mixed_compliance", "cctv", "close_up", "highway"]
sample_images = []

for cat in sample_categories:
    cat_dir = IMAGES_DIR / cat
    if cat_dir.exists():
        imgs = sorted(cat_dir.glob("*.jpg")) + sorted(cat_dir.glob("*.png")) + sorted(cat_dir.glob("*.webp"))
        if imgs:
            sample_images.append(imgs[0])

print(f"Running compliance on {len(sample_images)} sample images...\n")

for img_path in sample_images:
    results = model.predict(str(img_path), conf=0.25, verbose=False)
    print(f"--- {img_path.parent.name}/{img_path.name} ---")
    visualize_compliance(str(img_path), results)

### Aggregate Compliance Report

Let's compute compliance across ALL validation images and break it down by
scene category. This is what the safety manager would see in a dashboard.

In [ ]:
# ── Aggregate compliance across all images ────────────────────────────────────

all_assessments = []
all_image_paths = sorted(IMAGES_DIR.rglob("*.jpg")) + sorted(IMAGES_DIR.rglob("*.png")) + sorted(IMAGES_DIR.rglob("*.webp"))

# Deduplicate and sort
seen = set()
unique_images = []
for p in all_image_paths:
    if p.name not in seen:
        seen.add(p.name)
        unique_images.append(p)
unique_images.sort()

print(f"Processing {len(unique_images)} images...")

for img_path in unique_images:
    results = model.predict(str(img_path), conf=0.25, verbose=False)
    assessment = assess_compliance(results)
    assessment["image"] = img_path.name
    assessment["category"] = img_path.parent.name
    all_assessments.append(assessment)

# Overall summary
total_persons = sum(r["total_persons"] for r in all_assessments)
total_compliant = sum(r["compliant"] for r in all_assessments)
total_non_compliant = sum(r["non_compliant"] for r in all_assessments)

print(f"\n{'='*60}")
print(f"AGGREGATE COMPLIANCE REPORT")
print(f"{'='*60}")
print(f"Total workers detected:        {total_persons}")
print(f"Compliant (wearing hardhat):   {total_compliant}")
print(f"Non-compliant (no hardhat):    {total_non_compliant}")
print(f"Overall violation rate:         {total_non_compliant / max(total_persons, 1) * 100:.1f}%")
print(f"{'='*60}")

# Per-category breakdown
by_cat = defaultdict(lambda: {"persons": 0, "compliant": 0, "non_compliant": 0})
for r in all_assessments:
    by_cat[r["category"]]["persons"] += r["total_persons"]
    by_cat[r["category"]]["compliant"] += r["compliant"]
    by_cat[r["category"]]["non_compliant"] += r["non_compliant"]

print(f"\n{'Category':<22} {'Persons':>8} {'Compliant':>10} {'Violations':>10} {'Rate':>8}")
print("-" * 62)
for cat in sorted(by_cat):
    d = by_cat[cat]
    rate = d["non_compliant"] / max(d["persons"], 1) * 100
    print(f"{cat:<22} {d['persons']:>8} {d['compliant']:>10} {d['non_compliant']:>10} {rate:>7.1f}%")

---
### Real-World Consideration: Distance Filtering

In production, not every detected person is equally useful for compliance assessment.
Workers far from the camera appear as small bounding boxes where it's physically
impossible to determine if they're wearing a hardhat -- the helmet is just a few pixels.

**The pragmatic approach:** Rather than trying to assess compliance for everyone
(and getting unreliable results for distant workers), set a minimum person-size
threshold and only assess compliance for workers above that threshold.

This is a real-world lesson: **it's not your job to take requirements at face value
and burn out trying to meet them.** Understand what's feasible. Create a system
with guardrails that works reliably for a subset of cases. Negotiate the requirements
or find a different angle.

> **Discussion point:** What minimum person height (in pixels) would you choose?
> Consider: at 640px input, a 100px person is barely identifiable. A 200px person
> is clearly visible. We'll test a few thresholds and see which works best.

In [ ]:
# ── Person-size filtering: proxy for distance ────────────────────────────────
# Filter compliance assessment to only consider persons above a minimum pixel height.
# This is a proxy for distance -- smaller persons are further away and harder to assess.

def assess_compliance_filtered(results, conf=0.25, min_person_height_px=100,
                                head_fraction=0.4, iou_threshold=0.1):
    """Assess compliance only for persons above a minimum pixel height.

    Args:
        results: Ultralytics prediction results for one image
        conf: confidence threshold
        min_person_height_px: minimum person bbox height in pixels to consider
        head_fraction: top fraction of person bbox considered as head region
        iou_threshold: minimum IoU to consider a hardhat as 'worn'

    Returns:
        dict with compliance stats + filtered_out count
    """
    boxes = results[0].boxes
    persons, hardhats = [], []
    filtered_out = 0

    for i, cls_id in enumerate(boxes.cls.cpu().numpy()):
        if boxes.conf[i].item() < conf:
            continue
        xyxy = boxes.xyxy[i].cpu().numpy()
        if int(cls_id) == 0:   # hardhat
            hardhats.append(xyxy)
        elif int(cls_id) == 1: # person
            height_px = xyxy[3] - xyxy[1]
            if height_px >= min_person_height_px:
                persons.append(xyxy)
            else:
                filtered_out += 1

    compliant_flags = []
    for person in persons:
        head_region = [
            person[0],
            person[1],
            person[2],
            person[1] + (person[3] - person[1]) * head_fraction
        ]
        is_compliant = any(
            compute_iou(head_region, hh) >= iou_threshold for hh in hardhats
        )
        compliant_flags.append(is_compliant)

    total = len(persons)
    compliant_count = sum(compliant_flags)
    non_compliant = total - compliant_count
    rate = (non_compliant / total * 100) if total > 0 else 0

    return {
        "total_persons": total,
        "compliant": compliant_count,
        "non_compliant": non_compliant,
        "compliance_violation_rate": rate,
        "filtered_out": filtered_out,
        "persons": persons,
        "hardhats": hardhats,
        "compliant_flags": compliant_flags,
    }


# ── Compare compliance metrics across different height thresholds ─────────────

thresholds_px = [0, 50, 100, 150, 200]  # 0 = no filter

print(f"{'Threshold (px)':<16} {'Persons':<10} {'Filtered':<10} {'Compliant':<12} {'Non-compl':<12} {'Violation%':<12}")
print("-" * 72)

threshold_results = {}
for thresh in thresholds_px:
    total_p = total_c = total_nc = total_filt = 0

    for img_path in unique_images:
        results = model.predict(str(img_path), conf=0.25, verbose=False)
        assessment = assess_compliance_filtered(results, min_person_height_px=thresh)
        total_p += assessment["total_persons"]
        total_c += assessment["compliant"]
        total_nc += assessment["non_compliant"]
        total_filt += assessment["filtered_out"]

    rate = total_nc / max(total_p, 1) * 100
    threshold_results[thresh] = {
        "persons": total_p, "compliant": total_c,
        "non_compliant": total_nc, "filtered": total_filt, "rate": rate
    }
    print(f"{thresh:<16} {total_p:<10} {total_filt:<10} {total_c:<12} {total_nc:<12} {rate:<12.1f}")

# Plot violation rate vs threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

rates = [threshold_results[t]["rate"] for t in thresholds_px]
persons_kept = [threshold_results[t]["persons"] for t in thresholds_px]

ax1.plot(thresholds_px, rates, "o-", color="#e74c3c", linewidth=2, markersize=8)
ax1.set_xlabel("Min person height (px)")
ax1.set_ylabel("Violation rate (%)")
ax1.set_title("Compliance Violation Rate vs Distance Filter")
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(thresholds_px)), persons_kept, color="#3498db", edgecolor="white")
ax2.set_xticks(range(len(thresholds_px)))
ax2.set_xticklabels([str(t) for t in thresholds_px])
ax2.set_xlabel("Min person height (px)")
ax2.set_ylabel("Persons assessed")
ax2.set_title("Coverage vs Distance Filter")
ax2.grid(True, alpha=0.3, axis="y")

fig.suptitle("Distance Filtering Trade-off", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n★ Key insight: Distance filtering trades coverage for reliability.")
print("  A threshold of ~100px removes the hardest cases while keeping most workers.")
print("  This is the pragmatic approach: reliable results for a defined operating range.")

> **Takeaway:** In real deployments, you'll negotiate operating constraints.
> "We guarantee 95% compliance detection accuracy for workers within 30 meters
> of the camera" is a much stronger claim than "We try to detect everyone
> regardless of distance, but accuracy drops to 60% for distant workers."
>
> **The golden rule applies here too:** Use the model to detect objects (hardhats, persons).
> Use code to enforce operational constraints (minimum size, confidence thresholds, zones).

In [ ]:
# ── Compliance bar chart by category ──────────────────────────────────────────

cats_sorted = sorted(by_cat.keys())
compliant_vals = [by_cat[c]["compliant"] for c in cats_sorted]
non_compliant_vals = [by_cat[c]["non_compliant"] for c in cats_sorted]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(cats_sorted))
width = 0.35

bars1 = ax.bar(x - width/2, compliant_vals, width, label="Compliant",
               color="#27ae60", edgecolor="white")
bars2 = ax.bar(x + width/2, non_compliant_vals, width, label="Non-compliant",
               color="#e74c3c", edgecolor="white")

ax.set_ylabel("Number of workers")
ax.set_title("PPE Compliance by Scene Category", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cats_sorted, rotation=45, ha="right")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Add violation rate labels on top
for i, cat in enumerate(cats_sorted):
    total = by_cat[cat]["persons"]
    if total > 0:
        rate = by_cat[cat]["non_compliant"] / total * 100
        max_val = max(compliant_vals[i], non_compliant_vals[i])
        ax.text(i, max_val + 0.5, f"{rate:.0f}%", ha="center", fontsize=9,
                color="#e74c3c", fontweight="bold")

plt.tight_layout()
plt.show()

### Inference Speed: Foundation Model vs Student

The whole point of distillation is speed. Let's verify our student model
meets the 30+ FPS production requirement.

In [ ]:
# ── Inference speed comparison ────────────────────────────────────────────────
import time

# Pick 10 images for the benchmark
benchmark_images = unique_images[:10] if 'unique_images' in dir() else sorted(IMAGES_DIR.rglob("*.jpg"))[:10]

# -- Student model (YOLO26n) --
student = YOLO(str(MODEL_PATH))

# Warmup
for _ in range(3):
    student.predict(str(benchmark_images[0]), verbose=False)

# Timed run
start = time.perf_counter()
for img_path in benchmark_images:
    student.predict(str(img_path), verbose=False)
student_time = (time.perf_counter() - start) / len(benchmark_images) * 1000  # ms

# -- Foundation model (YOLOe-26n-seg) --
yoloe_path = Path("yoloe-26n-seg.pt")
if not yoloe_path.exists():
    yoloe_path = WORKSHOP_DIR / "notebooks" / "yoloe-26n-seg.pt"

foundation_time = None
if yoloe_path.exists():
    foundation = YOLO(str(yoloe_path))
    foundation.set_classes(["helmet", "person"])

    # Warmup
    for _ in range(3):
        foundation.predict(str(benchmark_images[0]), verbose=False)

    # Timed run
    start = time.perf_counter()
    for img_path in benchmark_images:
        foundation.predict(str(img_path), verbose=False)
    foundation_time = (time.perf_counter() - start) / len(benchmark_images) * 1000

    del foundation

print(f"{'='*50}")
print(f"INFERENCE SPEED COMPARISON")
print(f"{'='*50}")
print(f"Student (YOLO26n):     {student_time:.1f} ms/image  ({1000/student_time:.0f} FPS)")
if foundation_time:
    print(f"Foundation (YOLOe):    {foundation_time:.1f} ms/image  ({1000/foundation_time:.0f} FPS)")
    speedup = foundation_time / student_time
    print(f"Speedup:               {speedup:.1f}x")
print(f"{'='*50}")

meets_target = student_time < 33.3  # 30 FPS = 33.3 ms
print(f"\n30+ FPS target: {'MET' if meets_target else 'NOT MET'} ({1000/student_time:.0f} FPS)")

---
## Part 6: Key Takeaways

### The Five Insights

| # | Insight | Evidence |
|---|---------|----------|
| 1 | **Label quality > data quantity** | Saturating curve (R^2=0.97) shows mAP50 asymptote at 0.527 with noisy labels. Better prompts broke through to 0.633. |
| 2 | **Prompt engineering is first-order** | `"helmet. person."` finds 2.2x more objects than verbose alternatives. Same phenomenon as LLM prompt sensitivity. |
| 3 | **2-class + post-processing beats 3-class** | Detecting absence (`no_hardhat`) has 25.5% recall. Detect objects, derive compliance with code. |
| 4 | **Tiny label filtering matters** | Removing labels below 3% normalized dimension cuts 35.6% of noise for +2.7% mAP improvement. |
| 5 | **Model metrics != business metrics** | mAP50 measures detection quality. Compliance accuracy measures the business outcome. A model with lower mAP can have better compliance accuracy if it excels at the specific detections that matter. |

### The Golden Rule

> **Detect THINGS with models, check RELATIONSHIPS with code.**
>
> Models are excellent at finding objects (hardhats, persons).
> Logic about how objects relate to each other ("is this person wearing that helmet?")
> belongs in your post-processing code, not in the model.

### The Pipeline Pattern

This workshop demonstrated a general pattern for building production CV systems:

```
Foundation Model (teacher)     -->  Label data automatically
Data Curation (human + code)   -->  Clean and validate labels
Fast Model (student)           -->  Train for production speed
Post-Processing (code)         -->  Derive business metrics from detections
```

This pattern applies far beyond PPE detection:
- **Manufacturing defect inspection**: Foundation model labels defects, student model runs on edge devices
- **Retail analytics**: Foundation model labels products/people, code computes shelf compliance or foot traffic
- **Agriculture**: Foundation model labels crop health indicators, student runs on drones

### Resources

- [MIT Data-Centric AI Course](https://dcai.csail.mit.edu/) -- Labs on label quality, data-centric evaluation
- [FiftyOne Documentation](https://docs.voxel51.com/) -- Data-centric CV toolkit
- [Ultralytics Documentation](https://docs.ultralytics.com/) -- YOLO training and deployment
- Workshop notebooks: `demo_05_model_export.ipynb` (ONNX export), `demo_06_model_serving.ipynb` (FastAPI/LitServe)

### Pipeline Summary

| Stage | Tool | Key Finding |
|-------|------|-------------|
| Zero-shot baseline | YOLOe-26n-seg | Decent but slow, inconsistent on hardhats |
| Auto-label | SAM3 HF (single-concept prompts) | Prompt choice matters — always test variations |
| Data curation | `filter_tiny_labels.py` | Removing noise labels improves model quality |
| Train student | YOLO26n on curated labels | Fast inference, production-ready |
| Compliance | 2-class detection + post-processing | Detect things with models, check relationships with code |

> **Run this notebook top-to-bottom to see actual metrics.**
> The numbers above are qualitative — exact mAP50, speed, and compliance rates
> depend on your specific auto-labeling run, filtering, and training.

---

*End of Solution Walkthrough*